In [1]:
pip install pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install prophet

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/13.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.3 MB 8.5 MB/s eta 0:00:02
   - -------------------------------------- 0.5/13.3 MB 8.5 MB/s eta 0:00:02
   - -------------------------------------- 0.5/13.3 MB 8.5 MB/s eta 0:00:02
   --------- ------------------------------ 3.1/13.3 MB 3.6 MB/s eta 0:00:03
   ------------- -------------------------- 4.5/13.3 MB 4.1 MB/s eta 0:00:03
   ------------------ --------------------- 6.3/13.3 MB 5.0 MB/s eta 0:00:02
   --------------------- ------------------ 7.1/13.3 MB 4.9 MB/s eta 0:00:02
   ---------------------- ----------------- 7.3/13.3 MB 4.5 MB/s eta 0:00:02
   ------------------------ --------------- 8.1/13.3 MB 4.4 MB/s eta 0:00:02
   ------------------------- -------------- 8.4/13.3 MB 4.2 MB/s eta 0:00:02
   --------------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
pip install plotly

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.8 MB 7.5 MB/s eta 0:00:02
   ----- ---------------------------------- 1.3/9.8 MB 7.5 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.8 MB 2.9 MB/s eta 0:00:03
   ------------- -------------------------- 3.4/9.8 MB 4.1 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/9.8 MB 6.2 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.8 MB 6.3 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 6.1 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- ------------------- 1/2 [plotly]
   -------------------- -------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:


import pandas as pd
from prophet import Prophet
from pymongo import MongoClient
from bson import ObjectId
from datetime import datetime
import logging

logger = logging.getLogger(__name__)


# MongoDB — setup

client = MongoClient("mongodb://localhost:27017/")
db = client["buygenix"]

amazon_col   = db["amazon_products"]
flipkart_col = db["flipkart_products"]


def _get_collection(platform):
    """Platform ke hisaab se sahi collection return karo"""
    if platform == "flipkart":
        return flipkart_col
    return amazon_col  # default amazon


# Fetch product from correct collection

def get_price_history(product_id, platform=None):
    """
    Product dhundo — pehle platform specific collection mein,
    phir dono mein search karo agar platform nahi pata
    """
    product = None

    # if platform exist

    if platform:
        col = _get_collection(platform)
        try:
            product = col.find_one({"_id": ObjectId(product_id)})
        except Exception:
            pass
        if not product:
            product = col.find_one({"asin": product_id}) or \
                      col.find_one({"product_id": product_id})

    # if platform doesn't exist

    if not product:
        for col in [amazon_col, flipkart_col]:
            try:
                product = col.find_one({"_id": ObjectId(product_id)})
            except Exception:
                pass
            if not product:
                product = col.find_one({"asin": product_id}) or \
                          col.find_one({"product_id": product_id})
            if product:
                break

    if not product:
        return None, None

    return product, product.get("price_history", [])



# Prepare DataFrame

def prepare_dataframe(price_history):
    records = []
    for entry in price_history:
        price = entry.get("price")
        timestamp = entry.get("timestamp")
        if price and timestamp:
            records.append({"ds": timestamp, "y": float(price)})

    if not records:
        return None

    df = pd.DataFrame(records)
    df["ds"] = pd.to_datetime(df["ds"])
    df = df.sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    return df

# Prophet running

def run_prophet(df, periods=30):
    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=len(df) >= 14,
        daily_seasonality=False,
        changepoint_prior_scale=0.3,
    )
    model.fit(df)
    future = model.make_future_dataframe(periods=periods, freq="D")
    return model.predict(future)


# prediction

def predict_price(product_id, platform=None, periods=30):
    """
    Full pipeline:
    product_id = MongoDB _id / asin / flipkart product_id
    platform   = "amazon" / "flipkart" / None (auto-detect)
    """
    product, price_history = get_price_history(product_id, platform)

    if not product:
        return {"error": "Product not found"}

    if len(price_history) < 2:
        return {
            "error": "Not enough price history. Need at least 2 data points.",
            "data_points": len(price_history),
            "current_price": product.get("current_price"),
            "title": product.get("title"),
        }

    df = prepare_dataframe(price_history)
    if df is None or len(df) < 2:
        return {"error": "Could not prepare data for prediction"}

    try:
        forecast = run_prophet(df, periods)
    except Exception as e:
        logger.error(f"Prophet failed: {e}")
        return {"error": f"Prediction failed: {str(e)}"}

    current_price = product.get("current_price", df["y"].iloc[-1])
    last_known_date = df["ds"].max()
    future_forecast = forecast[forecast["ds"] > last_known_date].copy()

    # Next 7 days
    next_7 = future_forecast.head(7)[["ds", "yhat"]].copy()
    next_7["ds"] = next_7["ds"].dt.strftime("%Y-%m-%d")
    next_7["yhat"] = next_7["yhat"].round(2)

    # Full forecast
    full_forecast = [
        {
            "date": row["ds"].strftime("%Y-%m-%d"),
            "predicted_price": round(max(row["yhat"], 0), 2),
            "lower_bound": round(max(row["yhat_lower"], 0), 2),
            "upper_bound": round(row["yhat_upper"], 2),
        }
        for _, row in future_forecast.iterrows()
    ]

    # Trend
    predicted_end = future_forecast["yhat"].iloc[-1] if not future_forecast.empty else current_price

    if predicted_end < current_price * 0.97:
        trend, recommendation = "falling", "wait"
        reason = f"Price {round((1 - predicted_end/current_price)*100, 1)}% girne ki ummeed hai"
    elif predicted_end > current_price * 1.03:
        trend, recommendation = "rising", "buy_now"
        reason = f"Price {round((predicted_end/current_price - 1)*100, 1)}% badhne ki ummeed hai"
    else:
        trend, recommendation = "stable", "buy_now"
        reason = "Price stable rahegi"

    # Best buy date
    if not future_forecast.empty:
        min_idx = future_forecast["yhat"].idxmin()
        best_buy_date  = future_forecast.loc[min_idx, "ds"].strftime("%Y-%m-%d")
        best_buy_price = round(future_forecast.loc[min_idx, "yhat"], 2)
    else:
        best_buy_date  = datetime.now().strftime("%Y-%m-%d")
        best_buy_price = current_price

    return {
        "product": {
            "id": str(product["_id"]),
            "title": product.get("title"),
            "platform": product.get("platform"),
            "current_price": current_price,
            "url": product.get("url"),
        },
        "prediction": {
            "trend": trend,
            "recommendation": recommendation,
            "reason": reason,
            "best_buy_date": best_buy_date,
            "best_buy_price": best_buy_price,
            "predicted_min": round(float(future_forecast["yhat"].min()), 2) if not future_forecast.empty else current_price,
            "predicted_max": round(float(future_forecast["yhat"].max()), 2) if not future_forecast.empty else current_price,
            "next_7_days": next_7.to_dict("records"),
            "full_30_day_forecast": full_forecast,
            "data_points_used": len(df),
        }
    }


def get_predictable_products():
    
    pipeline = [
        {
            "$project": {
                "title": 1,
                "platform": 1,
                "current_price": 1,
                "asin": 1,
                "product_id": 1,
                "history_count": {"$size": {"$ifNull": ["$price_history", []]}}
            }
        },
        {"$match": {"history_count": {"$gte": 2}}},
        {"$sort": {"history_count": -1}}
    ]

    amazon_results   = list(amazon_col.aggregate(pipeline))
    flipkart_results = list(flipkart_col.aggregate(pipeline))

    
    for p in amazon_results:
        p["_source"] = "amazon_products"
    for p in flipkart_results:
        p["_source"] = "flipkart_products"

    return amazon_results + flipkart_results



if __name__ == "__main__":
    print("Checking products with price history...\n")

    # Amazon
    amazon_count = amazon_col.count_documents({})
    flipkart_count = flipkart_col.count_documents({})
    print(f"amazon_products   : {amazon_count} documents")
    print(f"flipkart_products : {flipkart_count} documents")
    print()

    predictable = get_predictable_products()

    if not predictable:
        print("No products with enough price history yet.")
        print("Scrapers ek baar aur run karo.")
    else:
        print(f"Found {len(predictable)} predictable products:\n")
        for p in predictable[:5]:
            print(f"  [{p['platform']}] {p['title'][:55]} — {p['history_count']} entries — Rs.{p['current_price']}")

        first = predictable[0]
        print(f"\nRunning prediction for: {first['title'][:55]}")
        print("-" * 60)

        result = predict_price(str(first["_id"]), platform=first.get("platform"))

        if "error" in result:
            print(f"Error: {result['error']}")
        else:
            pred = result["prediction"]
            print(f"Platform      : {result['product']['platform']}")
            print(f"Current Price : Rs.{result['product']['current_price']}")
            print(f"Trend         : {pred['trend']}")
            print(f"Recommendation: {pred['recommendation']}")
            print(f"Reason        : {pred['reason']}")
            print(f"Best Buy Date : {pred['best_buy_date']} @ Rs.{pred['best_buy_price']}")
            print(f"\nNext 7 Days Forecast:")
            for day in pred["next_7_days"]:
                print(f"  {day['ds']}  ->  Rs.{day['yhat']}")

Checking products with price history...

amazon_products   : 10 documents
flipkart_products : 10 documents

Found 20 predictable products:

  [amazon] acer SmartChoice Aspire Lite, AMD Ryzen 5-5625U Process — 2 entries — Rs.41990.0
  [amazon] acer Aspire Lite, AMD Ryzen 3-5300U, 8 GB RAM, 512 GB S — 2 entries — Rs.35990.0
  [amazon] Dell 15, AMD Ryzen 7-7730U, 16GB DDR4, 512GB SSD, FHD,  — 2 entries — Rs.51990.0
  [amazon] BrowseBook 14.1" FHD IPS Laptop | Best Student & Office — 2 entries — Rs.12990.0
  [amazon] HP 15, 13th Gen Intel Core i5-1335U (16GB DDR4,512GB SS — 2 entries — Rs.55490.0

Running prediction for: acer SmartChoice Aspire Lite, AMD Ryzen 5-5625U Process
------------------------------------------------------------
Platform      : amazon
Current Price : Rs.41990.0
Trend         : stable
Recommendation: buy_now
Reason        : Price stable rahegi
Best Buy Date : 2026-03-11 @ Rs.41990.0

Next 7 Days Forecast:
  2026-03-11  ->  Rs.41990.0
  2026-03-12  ->  Rs.41990.0
  20